In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import map_coordinates
from scipy.ndimage import convolve1d
from scipy.ndimage import maximum_filter
import cv26_lab2_utils as p

In [30]:
# Creating images paths with correct order 
image = [img for img in os.listdir("cv26_lab2_part1")]
image.sort(key=lambda img: int(img.split('.')[0]))
image_path = [os.path.join("cv26_lab2_part1", img) for img in image]

In [51]:
def lk(I1, I2, features, rho, epsilon, d_x0=None, d_y0=None):

    # Get coords of features
    x_feat = features[:, 0]
    y_feat = features[:, 1]

    # Initialize d = [0, 0]
    n_pts = features.shape[0]
    if d_x0 is None:
        dx = np.zeros(n_pts, dtype=np.float32)
    else:
        dx = d_x0.copy() 

    if d_y0 is None:
        dy = np.zeros(n_pts, dtype=np.float32)
    else:
        dy = d_y0.copy()

    # Mean d for crop
    dx_mean = np.mean(dx)
    dy_mean = np.mean(dy)
    
    # Derivatives of crop
    I1_x = cv2.Sobel(I1, cv2.CV_32F, 1, 0, ksize=3)
    I1_y = cv2.Sobel(I1, cv2.CV_32F, 0, 1, ksize=3)
    
    # Create Guassian filtrer G_rho
    kernel_size = int(np.ceil(3 * rho) * 2 + 1)
    G_1d = cv2.getGaussianKernel(kernel_size, rho)
    G_rho = G_1d @ G_1d.T
    
    # Create meshgrid for crop
    H, W = I1.shape
    X_grid, Y_grid = np.meshgrid(np.arange(W), np.arange(H))
    X_grid = X_grid.astype(np.float32)
    Y_grid = Y_grid.astype(np.float32)
    
    for i in range(100):

        x_coords_dense = X_grid + dx_mean
        y_coords_dense = Y_grid + dy_mean
        
        # Warping I1 and derivatives at new dense coords
        I1_warped = map_coordinates(I1, [y_coords_dense, x_coords_dense], order=1)
        I1_x_warped = map_coordinates(I1_x, [y_coords_dense, x_coords_dense], order=1)
        I1_y_warped = map_coordinates(I1_y, [y_coords_dense, x_coords_dense], order=1)
        
        # Claculate error for all crop
        E = I2 - I1_warped
     
        # Prepare terms for convolution
        Ix2 = I1_x_warped**2
        Iy2 = I1_y_warped**2
        Ixy = I1_x_warped * I1_y_warped
        IxE = I1_x_warped * E
        IyE = I1_y_warped * E
        
        # Convolution of crop with gaussian filter
        A11_dense = cv2.filter2D(Ix2, -1, G_rho) + epsilon
        A22_dense = cv2.filter2D(Iy2, -1, G_rho) + epsilon
        A12_dense = cv2.filter2D(Ixy, -1, G_rho)
        b1_dense = cv2.filter2D(IxE, -1, G_rho)
        b2_dense = cv2.filter2D(IyE, -1, G_rho)
        
        # Sample the result of convolution to features coords
        m11 = map_coordinates(A11_dense, [y_feat, x_feat], order=1)
        m22 = map_coordinates(A22_dense, [y_feat, x_feat], order=1)
        m12 = map_coordinates(A12_dense, [y_feat, x_feat], order=1)
        b1 = map_coordinates(b1_dense, [y_feat, x_feat], order=1)
        b2 = map_coordinates(b2_dense, [y_feat, x_feat], order=1)
        
        # Cramer solution
        det = m11 * m22 - m12**2
        det_safe = det + 1e-9
        
        ux = (m22 * b1 - m12 * b2) / det_safe
        uy = (-m12 * b1 + m11 * b2) / det_safe
        
        # Update optical flow
        dx = dx + ux
        dy = dy + uy
        
        # Update mean d for crop
        dx_mean = np.mean(dx)
        dy_mean = np.mean(dy)
        
        # Check threshold
        #if np.max(np.abs(ux)) < 0.005 and np.max(np.abs(uy)) < 0.005:
            #break
            
    return dx, dy

In [33]:
def displ(dx, dy, method='threshold', energy_threshold=0.005):
    
    if method == 'threshold':

        energy = dx**2 + dy**2
        mask = energy > energy_threshold
        
        if np.any(mask):
            displ_x = np.mean(dx[mask])
            displ_y = np.mean(dy[mask])
        else:
            displ_x, displ_y = 0.0, 0.0
            
    elif method == 'median':
        displ_x = np.median(dx)
        displ_y = np.median(dy)
        
    return np.array([displ_x, displ_y])

In [45]:
bounding_boxes = {
    "face": [154, 102, 67, 115],
    "left_hand": [93, 272, 56, 83],
    "right_hand": [201, 270, 56, 83]
}

output_dir = 'test_results'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

fig, ax = plt.subplots(figsize=(10, 8))

for t in range(len(image_path)-1):
    I_prev = (cv2.imread(image_path[t], cv2.IMREAD_GRAYSCALE) / 255.0).astype(np.float32)
    I_curr = (cv2.imread(image_path[t+1], cv2.IMREAD_GRAYSCALE) / 255.0).astype(np.float32)
    
    ax.clear()
    ax.imshow(I_curr, cmap='gray')
    ax.set_title(f"Frame {t:03d}")
    
    for name, box in bounding_boxes.items():
        x, y, w, h = map(int, box)
        crop1 = I_prev[y:y+h, x:x+w]
        crop2 = I_curr[y:y+h, x:x+w]
        
        feats = cv2.goodFeaturesToTrack(np.float32(crop1), 30, 0.1, 5)
        if feats is not None:
            feats = feats.reshape(-1, 2)
            
            # Lucas-Kanade
            dx, dy = lk(crop1, crop2, feats, rho=3.0, epsilon=0.05)
            
            ux_box, uy_box = displ(dx, dy, method='threshold')
            
            ax.quiver(feats[:, 0] + x, feats[:, 1] + y, -dx, -dy, color='r', angles='xy', scale=200)
            
            bounding_boxes[name][0] -= ux_box
            bounding_boxes[name][1] -= uy_box
            
            rect = plt.Rectangle((bounding_boxes[name][0], bounding_boxes[name][1]), w, h, fill=False, color='g')
            ax.add_patch(rect)
            ax.text(bounding_boxes[name][0], bounding_boxes[name][1]-5, name, color='lime', fontsize=9)
            
    save_path = os.path.join(output_dir, f"frame_{t:03d}.png")
    plt.savefig(save_path)
    print(f"Αποθηκεύτηκε: {save_path}")

plt.close(fig)
print("Completed")

Αποθηκεύτηκε: test_results\frame_000.png
Αποθηκεύτηκε: test_results\frame_001.png
Αποθηκεύτηκε: test_results\frame_002.png
Αποθηκεύτηκε: test_results\frame_003.png
Αποθηκεύτηκε: test_results\frame_004.png
Αποθηκεύτηκε: test_results\frame_005.png
Αποθηκεύτηκε: test_results\frame_006.png
Αποθηκεύτηκε: test_results\frame_007.png
Αποθηκεύτηκε: test_results\frame_008.png
Αποθηκεύτηκε: test_results\frame_009.png
Αποθηκεύτηκε: test_results\frame_010.png
Αποθηκεύτηκε: test_results\frame_011.png
Αποθηκεύτηκε: test_results\frame_012.png
Αποθηκεύτηκε: test_results\frame_013.png
Αποθηκεύτηκε: test_results\frame_014.png
Αποθηκεύτηκε: test_results\frame_015.png
Αποθηκεύτηκε: test_results\frame_016.png
Αποθηκεύτηκε: test_results\frame_017.png
Αποθηκεύτηκε: test_results\frame_018.png
Αποθηκεύτηκε: test_results\frame_019.png
Αποθηκεύτηκε: test_results\frame_020.png
Αποθηκεύτηκε: test_results\frame_021.png
Αποθηκεύτηκε: test_results\frame_022.png
Αποθηκεύτηκε: test_results\frame_023.png
Αποθηκεύτηκε: te

In [44]:
print(image_path)

['cv26_lab2_part1\\1.png', 'cv26_lab2_part1\\2.png', 'cv26_lab2_part1\\3.png', 'cv26_lab2_part1\\4.png', 'cv26_lab2_part1\\5.png', 'cv26_lab2_part1\\6.png', 'cv26_lab2_part1\\7.png', 'cv26_lab2_part1\\8.png', 'cv26_lab2_part1\\9.png', 'cv26_lab2_part1\\10.png', 'cv26_lab2_part1\\11.png', 'cv26_lab2_part1\\12.png', 'cv26_lab2_part1\\13.png', 'cv26_lab2_part1\\14.png', 'cv26_lab2_part1\\15.png', 'cv26_lab2_part1\\16.png', 'cv26_lab2_part1\\17.png', 'cv26_lab2_part1\\18.png', 'cv26_lab2_part1\\19.png', 'cv26_lab2_part1\\20.png', 'cv26_lab2_part1\\21.png', 'cv26_lab2_part1\\22.png', 'cv26_lab2_part1\\23.png', 'cv26_lab2_part1\\24.png', 'cv26_lab2_part1\\25.png', 'cv26_lab2_part1\\26.png', 'cv26_lab2_part1\\27.png', 'cv26_lab2_part1\\28.png', 'cv26_lab2_part1\\29.png', 'cv26_lab2_part1\\30.png', 'cv26_lab2_part1\\31.png', 'cv26_lab2_part1\\32.png', 'cv26_lab2_part1\\33.png', 'cv26_lab2_part1\\34.png', 'cv26_lab2_part1\\35.png', 'cv26_lab2_part1\\36.png', 'cv26_lab2_part1\\37.png', 'cv26_lab

In [49]:
def multi_scale_lk(I1, I2, features, rho, epsilon, n_scales):

    sigma = 3

    pyramid1 = [I1]
    pyramid2 = [I2]
    
    filtered1 = I1
    filtered2 = I2
    
    for i in range(1, n_scales):

        kernel_size = int(np.ceil(3 * sigma) * 2 + 1)
    
        G_1d = cv2.getGaussianKernel(kernel_size, sigma)
        G_s = G_1d @ G_1d.T

        filtered1 = cv2.filter2D(filtered1, -1, G_s)
        filtered2 = cv2.filter2D(filtered2, -1, G_s)

        pyramid1.append(cv2.resize(filtered1, (0, 0), fx=0.5, fy=0.5))
        pyramid2.append(cv2.resize(filtered2, (0, 0), fx=0.5, fy=0.5))
    
    dx = np.zeros(len(features), dtype=np.float32)
    dy = np.zeros(len(features), dtype=np.float32)
     
    for s in reversed(range(n_scales)):
        curr_I1 = pyramid1[s]
        curr_I2 = pyramid2[s]
        
        curr_features = features / 2**s
        
        dx, dy = lk(curr_I1, curr_I2, curr_features, rho, epsilon, dx, dy)
        
        # 4. Μεταφορά στην επόμενη (μεγαλύτερη) κλίμακα
        if s > 0:
            dx *= 2.0
            dy *= 2.0
            
    return dx, dy

In [54]:
bounding_boxes = {
    "face": [154, 102, 67, 115],
    "left_hand": [93, 272, 56, 83],
    "right_hand": [201, 270, 56, 83]
}

n_scales = 4 
rho = 2.0
epsilon = 0.05
output_dir = 'test_multiscale'
if not os.path.exists(output_dir): os.makedirs(output_dir)

fig, ax = plt.subplots(figsize=(10, 8))

for t in range(len(image_path) - 1):
    I_prev = cv2.imread(image_path[t], cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
    I_curr = cv2.imread(image_path[t+1], cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0
    
    ax.clear()
    ax.imshow(I_curr, cmap='gray')
    
    for name, box in bounding_boxes.items():
        x, y, w, h = map(int, box)
        crop1 = I_prev[y:y+h, x:x+w]
        crop2 = I_curr[y:y+h, x:x+w]
            
        feats = cv2.goodFeaturesToTrack(crop1, 30, 0.1, 5)
        if feats is not None:
            feats = feats.reshape(-1, 2)
            
            dx, dy = multi_scale_lk(crop1, crop2, feats, rho, epsilon, n_scales)
            
            ux_box, uy_box = displ(dx, dy, method='threshold')
            
            ax.quiver(feats[:,0]+x, feats[:,1]+y, -dx, -dy, color='r', angles='xy', scale_units='xy', scale=1)
            
            bounding_boxes[name][0] -= ux_box
            bounding_boxes[name][1] -= uy_box
            
            rect = plt.Rectangle((bounding_boxes[name][0], bounding_boxes[name][1]), w, h, fill=False, edgecolor='lime', linewidth=2)
            ax.add_patch(rect)

    plt.savefig(os.path.join(output_dir, f"frame_{t:03d}.png"))
    print(f"Frame {t} done.")

plt.close()

Frame 0 done.
Frame 1 done.
Frame 2 done.
Frame 3 done.
Frame 4 done.
Frame 5 done.
Frame 6 done.
Frame 7 done.
Frame 8 done.
Frame 9 done.
Frame 10 done.
Frame 11 done.
Frame 12 done.
Frame 13 done.
Frame 14 done.
Frame 15 done.
Frame 16 done.
Frame 17 done.
Frame 18 done.
Frame 19 done.
Frame 20 done.
Frame 21 done.
Frame 22 done.
Frame 23 done.
Frame 24 done.
Frame 25 done.
Frame 26 done.
Frame 27 done.
Frame 28 done.
Frame 29 done.
Frame 30 done.
Frame 31 done.
Frame 32 done.
Frame 33 done.
Frame 34 done.
Frame 35 done.
Frame 36 done.
Frame 37 done.
Frame 38 done.
Frame 39 done.
Frame 40 done.
Frame 41 done.
Frame 42 done.
Frame 43 done.
Frame 44 done.
Frame 45 done.
Frame 46 done.
Frame 47 done.
Frame 48 done.
Frame 49 done.
Frame 50 done.
Frame 51 done.
Frame 52 done.
Frame 53 done.
Frame 54 done.
Frame 55 done.
Frame 56 done.
Frame 57 done.
Frame 58 done.
Frame 59 done.
Frame 60 done.
Frame 61 done.
Frame 62 done.
Frame 63 done.
Frame 64 done.
Frame 65 done.
Frame 66 done.
Frame

# Μέρος 2

## 2.1.1

In [13]:
def apply_3d_gaussian_filter(video, sigma_x, sigma_y, sigma_t):

    # Σ matrix is diagonal so we can break convolution with 3d gaussian filter to 3 convolutions with 1d gaussian filters
    kernel_x = cv2.getGaussianKernel(int(2 * np.ceil(3 * sigma_x) + 1), sigma_x).flatten()
    kernel_y = cv2.getGaussianKernel(int(2 * np.ceil(3 * sigma_y) + 1), sigma_y).flatten()
    if sigma_t:
        kernel_t = cv2.getGaussianKernel(int(2 * np.ceil(3 * sigma_t) + 1), sigma_t).flatten() 
    
    # Separate Convolutions with 1d gaussian filters
    L = convolve1d(input=video, weights=kernel_x, axis=1) # default mode reflect. Output size = Input size
    L = convolve1d(input=L, weights=kernel_y, axis=0)
    if sigma_t:
        L = convolve1d(input=L, weights=kernel_t, axis=2)
    return L

In [36]:
def get_interest_points(H, sigma, theta=0.005, N=500, method="Local-Maxima"):
    if method == "Local-Maxima":
        # Find max element in 3x3x3 neighborhood
        local_max = maximum_filter(H, size=3)
        
        # Keep local maxima greater than threshold
        cond = (H == local_max) & (H > theta*np.max(H))
        y_coords, x_coords, t_coords = np.where(cond)
    
    if method == "Top-N":    
        flat_H = H.flatten()
        top_indices = np.argsort(flat_H)[-N:][::-1]
        y_coords, x_coords, t_coords = np.unravel_index(top_indices, H.shape)
    
    points = np.zeros((len(y_coords), 4))
    points[:, 0] = x_coords      # x 
    points[:, 1] = y_coords      # y 
    points[:, 2] = t_coords      # t 
    points[:, 3] = sigma         # sigma
    
    return points

In [50]:
def harris_stephens(video, sigma=4, taph=1.5, k=0.005, s=2, theta=0.8):
    # Smoothing
    L = apply_3d_gaussian_filter(video, sigma, sigma, taph)
    
    # Kernels for derivatives
    derivative_kernel = np.array([-1, 0, 1]) / 2.0
    
    # Compute derivatives
    Lx = convolve1d(input=L, weights=derivative_kernel, axis=1)
    Ly = convolve1d(input=L, weights=derivative_kernel, axis=0)
    Lt = convolve1d(input=L, weights=derivative_kernel, axis=2)
    
    i_s = s * sigma
    i_t = s * taph

    # Elements of J matrix
    M11 = apply_3d_gaussian_filter(Lx**2, i_s, i_s, i_t)
    M22 = apply_3d_gaussian_filter(Ly**2, i_s, i_s, i_t)
    M33 = apply_3d_gaussian_filter(Lt**2, i_s, i_s, i_t)
    M12 = apply_3d_gaussian_filter(Lx*Ly, i_s, i_s, i_t)
    M13 = apply_3d_gaussian_filter(Lx*Lt, i_s, i_s, i_t)
    M23 = apply_3d_gaussian_filter(Ly*Lt, i_s, i_s, i_t)
    
    # Harris-Stephens Criterion
    det = (M11 * (M22 * M33 - M23**2) - M12 * (M12 * M33 - M13 * M23) + M13 * (M12 * M23 - M13 * M22))   
    trace = M11 + M22 + M33
    H = det - k * (trace**3)

    points = get_interest_points(H, sigma, theta=theta, N=600, method="Local-Maxima")
    
    return points

In [11]:
def gabor_detector(video, sigma, taph, theta):

    # X, Y smoothing. Time axis remains as it was
    I_smoothed = apply_3d_gaussian_filter(video, sigma, sigma, 0)
    
    # Creating gabor filter for time axis
    t_limit = int(np.ceil(2 * taph))
    t = np.arange(-t_limit, t_limit + 1) # Time boundaries
    w = 4.0 / taph
    gaussian_part = np.exp(-t**2 / (2 * taph**2)) 
    h_ev = np.cos(2 * np.pi * t * w) * gaussian_part
    h_od = np.sin(2 * np.pi * t * w) * gaussian_part
    
    # L1 normalization (sum of abs values = 1)
    h_ev /= np.sum(np.abs(h_ev))
    h_od /= np.sum(np.abs(h_od))
    
    # Convolution with gabor filters
    res_ev = convolve1d(I_smoothed, h_ev_kernel, axis=2)
    res_od = convolve1d(I_smoothed, h_od_kernel, axis=2)
    
    # Criterion
    H = res_ev**2 + res_od**2

    points = get_interest_points(H, sigma, method="Local-Maxima")
    
    return points

In [6]:
video = p.read_video("person01_running_d1_uncomp.avi", show=False)
print(video.shape)

(120, 160, 335)


In [51]:
points = harris_stephens(video, sigma=4, taph=1.5, k=0.005, s=2, theta=0.7)
p.show_detection(video_=video, points_=points, save_path="test_save")

KeyboardInterrupt: 